# BellaBox Product CNN — Final Colab Runbook

Notebook تسليم نهائي ومنظم لتشغيل نموذج CNN على Google Colab باستخدام ملف منتجات سلة Excel.

## مراحل التشغيل
1. ربط Google Drive والتحقق من المسارات.
2. فحص GPU وTensorFlow وتثبيت المتطلبات.
3. فحص Dataset الموجود: إذا كان موجودًا لن تتم إعادة بنائه أو تنزيل الصور من جديد.
4. بناء Dataset من Excel فقط إذا كان غير موجود.
5. فحص التقسيم Grouped حسب `product_id` لمنع تسريب المنتج.
6. تشغيل التدريب أو استئنافه من Checkpoints الموجودة في Google Drive.
7. التحقق من النموذج النهائي والتقارير وConfusion Matrix.
8. اختبار صورة جديدة وضغط نتائج التشغيل للتسليم.

> شغّل الخلايا بالترتيب. لا تحذف مجلد `outputs_final` إذا أردت استئناف التدريب بعد انقطاع Colab.

In [ ]:
from pathlib import Path
import csv
import json
import os
import shutil
import subprocess
import sys

from google.colab import drive
from IPython.display import display, Image as DisplayImage

drive.mount('/content/drive')

DRIVE_ROOT = Path('/content/drive/MyDrive/BellaBox_Product_CNN_Colab_Project')
ZIP_CANDIDATES = [
    DRIVE_ROOT / 'BellaBox_Product_CNN_Final_Project.zip',
    DRIVE_ROOT / 'BellaBox_Product_CNN_Colab_Project.zip',
]
ZIP_PATH = next((path for path in ZIP_CANDIDATES if path.exists()), None)
PROJECT_DIR = Path('/content/bellabox-product-cnn')
XLSX_PATH = DRIVE_ROOT / 'bellabox_products.xlsx'
DATA_DIR = DRIVE_ROOT / 'dataset'
OUTPUT_DIR = DRIVE_ROOT / 'outputs_final'

if not DRIVE_ROOT.exists():
    raise FileNotFoundError(f'مجلد المشروع غير موجود في Google Drive: {DRIVE_ROOT}')

if not PROJECT_DIR.exists():
    if ZIP_PATH is None:
        expected = ', '.join(str(path) for path in ZIP_CANDIDATES)
        raise FileNotFoundError(f'ملف ZIP غير موجود. المسارات المفحوصة: {expected}')
    subprocess.run(['unzip', '-q', '-o', str(ZIP_PATH), '-d', '/content'], check=True)
    print('تم فك ضغط المشروع من Google Drive.')
else:
    print('مجلد المشروع موجود في بيئة Colab؛ سيتم استخدامه كما هو.')

print('PROJECT_DIR:', PROJECT_DIR)
print('XLSX_PATH:', XLSX_PATH)
print('DATA_DIR:', DATA_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)

## المرحلة 1 — فحص بيئة Colab وتثبيت المتطلبات

التدريب المقصود على GPU. الإصدار الحالي من المتطلبات يستخدم TensorFlow 2.20 أو أحدث، وهو متوافق مع إصدارات Python الحديثة في Colab.

In [ ]:
%cd /content/bellabox-product-cnn

# Install only missing/incompatible requirements; do not force-upgrade Colab's scientific stack.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'ml/requirements-colab.txt'], check=True)
print('تم فحص المتطلبات بدون ترقية قسرية لحزم NumPy/SciPy/scikit-learn.')

import tensorflow as tf

print('TensorFlow:', tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print('GPUs:', gpus)
if not gpus:
    raise RuntimeError('لم يتم العثور على GPU. فعّل Runtime > Change runtime type > T4 GPU ثم أعد تشغيل الخلية.')
print('بيئة GPU جاهزة للتدريب.')

In [ ]:
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=False)
else:
    print('nvidia-smi غير متاح؛ اعتمد على فحص TensorFlow السابق.')

## المرحلة 2 — فحص Dataset وعدم إعادة بنائه دون حاجة

هذه الخلية تفحص `manifest.csv`، وملخص Dataset، ووجود الصور المحلية. إذا كان Dataset موجودًا وقابلًا للاستخدام، ستظهر رسالة واضحة وسيتم الانتقال مباشرة إلى التدريب والاختبار. لن يتم تنزيل الصور مرة أخرى.

In [ ]:
import pandas as pd

MANIFEST_PATH = DATA_DIR / 'manifest.csv'
SUMMARY_PATH = DATA_DIR / 'dataset_summary.json'

def dataset_is_ready(data_dir: Path) -> tuple[bool, str, pd.DataFrame | None]:
    manifest_path = data_dir / 'manifest.csv'
    if not manifest_path.exists():
        return False, 'manifest.csv غير موجود', None
    try:
        frame = pd.read_csv(manifest_path)
    except Exception as error:
        return False, f'تعذر قراءة manifest.csv: {error}', None
    required = {'product_id', 'label', 'local_path', 'download_status'}
    missing = required - set(frame.columns)
    if missing:
        return False, f'أعمدة ناقصة: {sorted(missing)}', frame
    usable = frame[frame['download_status'].astype(str).eq('ok')].copy()
    usable = usable[usable['local_path'].map(lambda value: Path(str(value)).exists())]
    if len(usable) == 0:
        return False, 'لا توجد صور محلية صالحة', frame
    if usable['label'].nunique() < 2:
        return False, 'يلزم وجود فئتين على الأقل', usable
    return True, 'جاهز', usable

dataset_ready, dataset_message, manifest = dataset_is_ready(DATA_DIR)
if dataset_ready:
    print('✅ Dataset للتدريب والاختبار موجودة في المسار المحدد.')
    print(f'سيتم استخدام {len(manifest)} صورة من {manifest.product_id.nunique()} منتجًا و{manifest.label.nunique()} فئة.')
    print('لن تتم إعادة بناء Dataset أو تنزيل الصور.')
else:
    print(f'⚠️ Dataset غير جاهزة: {dataset_message}')
    if not XLSX_PATH.exists():
        raise FileNotFoundError(f'ملف Excel غير موجود لبناء Dataset: {XLSX_PATH}')
    print('سيتم بناء Dataset مرة واحدة من ملف Excel الموجود في Google Drive.')
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    build_command = [
        sys.executable, 'ml/build_dataset.py',
        '--products-xlsx', str(XLSX_PATH),
        '--output-dir', str(DATA_DIR),
        '--category-level', '2',
        '--min-images-per-class', '20',
        '--min-products-per-class', '4',
        '--drop-small-classes',
        '--max-images-per-product', '3',
        '--download',
    ]
    subprocess.run(build_command, check=True)
    dataset_ready, dataset_message, manifest = dataset_is_ready(DATA_DIR)
    if not dataset_ready:
        raise RuntimeError(f'فشل تجهيز Dataset بعد البناء: {dataset_message}')
    print('✅ اكتمل بناء Dataset بنجاح.')

In [ ]:
if SUMMARY_PATH.exists():
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
    print(json.dumps(summary, ensure_ascii=False, indent=2))

print('توزيع الصور والمنتجات حسب الفئة:')
display(manifest.groupby('label').agg(
    images=('local_path', 'count'),
    products=('product_id', 'nunique'),
).sort_values('images', ascending=False))

## المرحلة 3 — فحص التقسيم ومنع تسريب المنتجات

يتم تقسيم الصور كمجموعات حسب `product_id`. لذلك لا يمكن أن تظهر صور المنتج نفسه في التدريب والتحقق والاختبار.

In [ ]:
import sys
sys.path.insert(0, str(PROJECT_DIR))
from ml import train as train_module

rows = train_module.read_manifest(DATA_DIR)
splits = train_module.grouped_stratified_split(rows, val_size=0.15, test_size=0.15)

for split_name, split_rows in splits.items():
    print(split_name, 'images=', len(split_rows), 'products=', len({row["product_id"] for row in split_rows}))

train_products = {row['product_id'] for row in splits['train']}
validation_products = {row['product_id'] for row in splits['validation']}
test_products = {row['product_id'] for row in splits['test']}
assert not train_products & validation_products
assert not train_products & test_products
assert not validation_products & test_products
print('✅ لا يوجد Product Leakage بين المجموعات.')

## المرحلة 4 — التدريب والاستئناف من Checkpoints

يحفظ التدريب النتائج داخل Google Drive. إذا انقطعت جلسة Colab، أعد تشغيل هذه الخلية بنفس `OUTPUT_DIR` مع `--resume`. لا تحذف المجلد.

الملفات الأساسية أثناء التدريب:
- `best.keras`: أفضل نموذج حسب `val_macro_f1`.
- `last.keras`: آخر نموذج محفوظ.
- `backup_stage1/` و`backup_stage2/`: حالة الاستئناف لكل مرحلة.
- `training_log.csv`: سجل Epochs.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FINAL_MODEL_PATH = OUTPUT_DIR / 'final_model.keras'
FORCE_RETRAIN = False

if FINAL_MODEL_PATH.exists() and not FORCE_RETRAIN:
    print('✅ النموذج النهائي موجود؛ سيتم تجاوز التدريب وإكمال مرحلة التحقق.')
    print('لتدريب جديد بالكامل استخدم مجلد نتائج جديد أو غيّر FORCE_RETRAIN إلى True.')
else:
    if any((OUTPUT_DIR / name).exists() for name in ['best.keras', 'last.keras', 'backup_stage1', 'backup_stage2']):
        print('♻️ توجد Checkpoints سابقة؛ سيحاول التدريب الاستئناف منها.')
    else:
        print('🚀 بدء تدريب جديد مع حفظ Checkpoints في Google Drive.')
    train_command = [
        sys.executable, 'ml/train.py',
        '--data-dir', str(DATA_DIR),
        '--output-dir', str(OUTPUT_DIR),
        '--epochs', '15',
        '--batch-size', '32',
        '--resume',
    ]
    subprocess.run(train_command, check=True)
    print('✅ انتهى أمر التدريب.')

## المرحلة 5 — التحقق من النموذج والتقارير

لا تعتبر التدريب مكتملًا حتى توجد الأوزان والتقارير التالية.

In [ ]:
required_outputs = [
    'best.keras', 'last.keras', 'final_model.keras', 'labels.json',
    'metrics.json', 'classification_report.txt', 'confusion_matrix.png',
    'training_curves.png', 'training_log.csv', 'run_config.json',
]
missing_outputs = [name for name in required_outputs if not (OUTPUT_DIR / name).exists()]
if missing_outputs:
    raise FileNotFoundError(f'ملفات نتائج ناقصة: {missing_outputs}')
print('✅ النموذج النهائي والتقارير الأساسية موجودة.')

metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
print('\nClassification Report:\n')
print((OUTPUT_DIR / 'classification_report.txt').read_text(encoding='utf-8'))

In [ ]:
display(DisplayImage(filename=str(OUTPUT_DIR / 'training_curves.png')))
display(DisplayImage(filename=str(OUTPUT_DIR / 'confusion_matrix.png')))

## المرحلة 6 — Confusion Matrix مُطبّعة وتقييم مستقل

المصفوفة المحفوظة من التدريب تعرض الأعداد. الخلية التالية تنشئ نسخة مُطبّعة حسب كل فئة، وتعرض تقريرًا منظمًا من مجموعة الاختبار نفسها.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

labels = sorted({row['label'] for row in rows})
label_to_id = {label: index for index, label in enumerate(labels)}
test_rows = splits['test']
test_data = train_module.make_dataset(test_rows, label_to_id, training=False, batch_size=32)
model = tf.keras.models.load_model(FINAL_MODEL_PATH, compile=False)
probabilities = model.predict(test_data, verbose=1)
y_true = np.array([label_to_id[row['label']] for row in test_rows])
y_pred = np.argmax(probabilities, axis=1)

report = train_module.classification_report_dict(y_true, y_pred, list(range(len(labels))))
display(pd.DataFrame(report).T.round(4))

cm = train_module.confusion_matrix(y_true, y_pred, labels=list(range(len(labels))))
row_totals = cm.sum(axis=1, keepdims=True)
normalized_cm = np.divide(cm.astype(float), row_totals, out=np.zeros_like(cm, dtype=float), where=row_totals != 0)

plt.figure(figsize=(12, 9))
sns.heatmap(normalized_cm, annot=True, fmt='.2f', cmap='Blues', vmin=0, vmax=1, xticklabels=labels, yticklabels=labels)
plt.title('BellaBox CNN — Normalized Confusion Matrix')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
normalized_path = OUTPUT_DIR / 'confusion_matrix_normalized.png'
plt.savefig(normalized_path, dpi=200, bbox_inches='tight')
plt.show()
print('تم حفظ:', normalized_path)

## المرحلة 7 — اختبار صورة جديدة (اختياري)

لإثبات الاستخدام العملي، ارفع صورة منتج لم تدخل في Dataset. يجب مقارنة الفئة المتوقعة مع الفئة الحقيقية من Excel.

In [ ]:
RUN_EXTERNAL_TEST = False
if RUN_EXTERNAL_TEST:
    from google.colab import files
    uploaded = files.upload()
    test_image = next(iter(uploaded))
    subprocess.run([
        sys.executable, 'ml/predict.py',
        '--model', str(FINAL_MODEL_PATH),
        '--labels', str(OUTPUT_DIR / 'labels.json'),
        '--image', test_image,
        '--top-k', '3',
    ], check=True)
else:
    print('اختبار الصورة الجديدة متوقف افتراضيًا. غيّر RUN_EXTERNAL_TEST إلى True لتفعيله.')

## المرحلة 8 — أرشفة نتائج التشغيل

بعد نجاح التدريب، يتم ضغط مجلد النتائج في Google Drive لحفظ نسخة التسليم.

In [ ]:
final_archive = shutil.make_archive(
    str(DRIVE_ROOT / 'BellaBox_CNN_Final_Run'),
    'zip',
    root_dir=OUTPUT_DIR,
)
print('✅ تم إنشاء أرشيف نتائج التشغيل:', final_archive)
print('المشروع:', ZIP_PATH)
print('Dataset:', DATA_DIR)
print('النموذج النهائي:', FINAL_MODEL_PATH)
print('التقارير:', OUTPUT_DIR)